# Collaborative Filtering

Memory-Based Algorithm
- **Item based: dot production 없는 버전 구현해볼 것**
  - 내적 안쓰는 Item based 방식 사용 많이 됨
- User based


Model-Based Algorithm
- Latent Factor 협업 필터링 방법 (Matrix Factorization)

# 구글 드라이브 연결

파일 > 드라이브 마운트 클릭

In [ ]:
import os
from pathlib import Path
DATA_DIR = Path("drive/MyDrive/kw/추천특강/강의자료/data/")
os.listdir(DATA_DIR)

# 데이터 로드

In [ ]:
MOVIE_DATA_PATH = f"{DATA_DIR}/movies_refined.csv"
RATING_DATA_PATH = f"{DATA_DIR}/ratings_refined.csv"

로드 후 데이터 타입 형변환  
id는 문자열로 처리

영화 데이터
```
movie_id: str
```

평점 데이터
```
user_id: str
movie_id: str
```

In [ ]:
import pandas as pd

movies_df = pd.read_csv(MOVIE_DATA_PATH,
    usecols=['movie_id', 'title'],
    dtype={'movie_id': str})

ratings_df = pd.read_csv(RATING_DATA_PATH,
    usecols=['user_id', 'movie_id', 'rating'],
    dtype={'user_id': str, 'movie_id': str})

In [ ]:
movie_ratings_df = pd.merge(ratings_df, movies_df, on='movie_id', how='left')
movie_ratings_df

,user_id,movie_id,rating,title
0,1,1,4.0,Toy Story (1995)
1,1,3,4.0,Grumpier Old Men (1995)
2,1,6,4.0,Heat (1995)
3,1,47,5.0,Seven (a.k.a. Se7en) (1995)
4,1,50,5.0,"Usual Suspects, The (1995)"
...,...,...,...,...
100784,610,166534,4.0,Split (2017)
100785,610,168248,5.0,John Wick: Chapter Two (2017)
100786,610,168250,5.0,Get Out (2017)
100787,610,168252,5.0,Logan (2017)


null 값 체크

In [ ]:
movie_ratings_df.columns[movie_ratings_df.isna().any()].tolist()

[]

영화명 결측치 체크

In [ ]:
movie_ratings_df[movie_ratings_df['title'].isnull()]

,user_id,movie_id,rating,title


# 영화 유사도 행렬 준비

유저X 입력되었을 때,

유저X가 본 영화 중 하나인 A 선택

이 영화A와 유사한 영화 찾을 때

사용할 행렬

In [ ]:
# 참고 (유저 유사도 행렬)
# user_movie_df = movie_ratings_df.pivot_table(values='rating', index='user_id', columns='title')
movie_user_df = movie_ratings_df.pivot_table(values='rating', index='title', columns='user_id')
movie_user_df

user_id,1,10,100,101,102,103,104,105,106,107,...,90,91,92,93,94,95,96,97,98,99
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
eXistenZ (1999),NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.5,NaN,NaN,NaN,NaN
xXx (2002),NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN
xXx: State of the Union (2005),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# rating 정보 있는 유저 개수
movie_ratings_df['user_id'].nunique()

610

In [ ]:
# 9685 x 610 행렬
# 사용자를 9685차원의 벡터로 보려는 것
movie_user_df.shape

(9685, 610)

## 결측치 처리

null값이 있으면 cosine similarity 함수가 안돌아감

하지만, null값을 0으로 치환하고 계산할경우 결과가 달라짐

(마치 해당 영화를 보고 0점을 준것으로 계산)

In [ ]:
tmp_movie_user_df = movie_user_df.copy().fillna(0)

## 유사도 행렬 계산

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

movie_similarity_matrix = cosine_similarity(tmp_movie_user_df)
movie_similarity_matrix.shape

(9685, 9685)

In [ ]:
movie_similarity_matrix

array([[1.        , 0.        , 0.        , ..., 0.32732684, 0.        ,
        0.        ],
       [0.        , 1.        , 0.70710678, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.70710678, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.32732684, 0.        , 0.        , ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ]])

## 데이터 프레임화

In [ ]:
movie_titles = movie_user_df.index
movie_titles

Index([''71 (2014)', ''Hellboy': The Seeds of Creation (2004)',
       ''Round Midnight (1986)', ''Salem's Lot (2004)',
       ''Til There Was You (1997)', ''Tis the Season for Love (2015)',
       ''burbs, The (1989)', ''night Mother (1986)',
       '(500) Days of Summer (2009)', '*batteries not included (1987)',
       ...
       'Zulu (2013)', '[REC] (2007)', '[REC]² (2009)',
       '[REC]³ 3 Génesis (2012)',
       'anohana: The Flower We Saw That Day - The Movie (2013)',
       'eXistenZ (1999)', 'xXx (2002)', 'xXx: State of the Union (2005)',
       '¡Three Amigos! (1986)', 'À nous la liberté (Freedom for Us) (1931)'],
      dtype='object', name='title', length=9685)

In [ ]:
# 유저와 유저 간에 대한 유사도
movie_similarity_df = pd.DataFrame(movie_similarity_matrix,
                                index=movie_titles, columns=movie_titles)
print(movie_similarity_df.shape)
movie_similarity_df.head()

(9685, 9685)


title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.141653,0.0,...,0.0,0.342055,0.543305,0.707107,0.0,0.0,0.139431,0.327327,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.0,1.000000,0.707107,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Round Midnight (1986),0.0,0.707107,1.000000,0.000000,0.000000,0.0,0.176777,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Salem's Lot (2004),0.0,0.000000,0.000000,1.000000,0.857493,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Til There Was You (1997),0.0,0.000000,0.000000,0.857493,1.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0


## 유사도 행렬 저장

인덱스가 1부터 시작하는 user_id 정보이므로

저장할 때 `index=True` 옵션 필요

In [ ]:
# 파일명에도 인덱스 포함 여부 명시하면 명확성 증가
movie_similarity_file_path = DATA_DIR / "movie_similarity_with_index.csv"
movie_similarity_file_path

PosixPath('drive/MyDrive/kw/추천특강/강의자료/data/movie_similarity_with_index.csv')

In [ ]:
# 영화 title로 사용하는 인덱스도 저장 필요한 정보
movie_similarity_df.to_csv(movie_similarity_file_path, index=True)
movie_similarity_df

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.141653,0.000000,...,0.000000,0.342055,0.543305,0.707107,0.0,0.000000,0.139431,0.327327,0.000000,0.0
'Hellboy': The Seeds of Creation (2004),0.000000,1.000000,0.707107,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
'Round Midnight (1986),0.000000,0.707107,1.000000,0.000000,0.000000,0.0,0.176777,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
'Salem's Lot (2004),0.000000,0.000000,0.000000,1.000000,0.857493,0.0,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
'Til There Was You (1997),0.000000,0.000000,0.000000,0.857493,1.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
eXistenZ (1999),0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.211467,0.216295,0.097935,0.132489,...,0.000000,0.000000,0.000000,0.000000,0.0,1.000000,0.192259,0.000000,0.170341,0.0
xXx (2002),0.139431,0.000000,0.000000,0.000000,0.000000,0.0,0.089634,0.000000,0.276512,0.019862,...,0.069716,0.305535,0.173151,0.246482,0.0,0.192259,1.000000,0.270034,0.100396,0.0
xXx: State of the Union (2005),0.327327,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.156764,0.000000,...,0.000000,0.382543,0.177838,0.231455,0.0,0.000000,0.270034,1.000000,0.000000,0.0


# Item-based CF

영화 유사도 기반 추천

(User-based CF는 사용자 유사도 기반 추천)

유저X 입력되었을 때,

유저X가 본 영화 중 하나인 A 선택

이 영화A와 유사한 영화 찾아서 추천

In [ ]:
# 샘플 사용자
user_id = "1"

# 샘플 사용자가 본 영화 중 임의로 한 개 선택
movie_title = (
    movie_ratings_df
    .loc[movie_ratings_df['user_id'] == user_id, 'title'][0]
)
movie_title

'Toy Story (1995)'

In [ ]:
# 사용자가 본 영화와 유사도가 높은 영화 10개 추천 함수
def get_recomendation(movie_similarity_df, title):
    return (
        movie_similarity_df[title]
        .sort_values(ascending=False)[1:11]
    )

In [ ]:
get_recomendation(movie_similarity_df, movie_title)

,Toy Story (1995)
title,
Toy Story 2 (1999),0.572601
Jurassic Park (1993),0.565637
Independence Day (a.k.a. ID4) (1996),0.564262
Star Wars: Episode IV - A New Hope (1977),0.557388
Forrest Gump (1994),0.547096
"Lion King, The (1994)",0.541145
Star Wars: Episode VI - Return of the Jedi (1983),0.541089
Mission: Impossible (1996),0.538913
Groundhog Day (1993),0.534169
